# v131 — Cross-Annotator 未注釈 Sibling 負例遮蔽 YOLO11m-seg 実験 (Idea 7)

**単一 YOLO Arm 学習のみ（Colab L4 約 1 時間 50 分）** — `v127` の完了済み対照群 `yolo_pilot_arm_a` と共通 in-split Refiner を読み取り専用で再利用し、保存先は `runs/v131_sibling_shield` に完全分離（他タブと同時実行可能）。

### 狙う数理ボトルネック（`v87` で実装断念されていた未踏機構）
- `prod` のレコード複製学習では、$R$ 人が注釈した同一画像上で $k$ 人が描いたフィラメントに対し、TAL 正例ソフトターゲット $t_+ \approx 0.50$ が $k$ 回与えられる一方、描かなかった残り $(R-k)$ 枚のコピーでは全アンカーに **背景ハード負例 ($t = 0, w = 1.0$)** が課されるため、停留確信度は
  $$p^* = \frac{k \cdot t_+}{k + (R - k) w_{\text{sib}}}$$
  へ収束する。
- $w_{\text{sib}} = 1.0$ (`prod`) では、**採点上の限界価値が明確に正（marginal ratio `0.65 >> 0.445`, `RESEARCH_LOG` §2.66）である $R=2, k=1$ 個体が $p^* \approx 0.25 < 0.26$ となって閾値直下で脱落**し、これが「primary FN の 57〜58% が `conf ∈ [0.02, 0.26)` に眠る」主因となっていた。
- `v87` はポリゴンを union 結合したため境界膨張で失敗したが、本実験では **正例ポリゴンを `prod` と 100% 同一の単一アノテータマスクに維持**したまま、同一画像の他アノテータが描いた非重複ポリゴン（`IoU ≤ 0.30`）を擬似クラス `cls=1` として Mosaic/Affine 変換に通し、未割当負例アンカーの BCE 重みのみ $w_{\text{sib}} = 0.20$ に緩和する（真の太陽面背景 $k=0$ は全コピーで $w=1.0$ を維持し、合意順序 $3/3 > 2/3 > 1/2 > 1/3 \gg 0/R$ も厳密保持）。


In [ ]:
# [1/7] 依存パッケージ・Drive マウント・実験設定 (v127 読み取り専用再利用 + 完全分離出力)
%pip install -q "ultralytics==8.4.120" pycocotools scipy
import hashlib, io, json, math, os, random, shutil, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm
from ultralytics import YOLO

drive.mount("/content/drive")

class CFG:
    SPLIT_MODE = "pilot"          # "pilot" (365 dev fit / 180 dev∩fold2 eval) or "temporal_confirm" (545 fit / 160 locked eval)
    SEED = 3407
    SIB_NEG_WEIGHT = 0.20         # 他アノテータのみが描いた領域の負例 BCE 重み (1.0 = prod 完全一致)
    SIB_OVERLAP_MAX_IOU = 0.30    # 自身のアノテーションと IoU <= 0.30 の他者個体のみ sibling_ignore 化
    SIB_DILATE_CELLS = 1
    PEER_WAIT_TIMEOUT_MIN = 240

    GROUPED_FOLDS_SHA256 = "febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e"
    TIME_BLOCK_SHA256 = "9a9b17252133c9500d44bf3b6ec56879a47e4b32d45cb14be080ba29148c732d"
    EXCLUSIONS_SHA256 = "a56372ea6493a6d3571910e60f291606b864a4f84a61baf1df475a6627270b0a"

    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    V127_EXP_DIR = DRIVE_ROOT / "runs" / "v127_spine_ab"

    EXP_DIR = DRIVE_ROOT / "runs" / "v131_sibling_shield"
    YOLO_DS_ROOT = DRIVE_ROOT / "yolo_datasets" / "v131_sibling_shield"
    LOCAL_ROOT = Path("/content/solar_v131")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"

    YOLO_MODEL = "yolo11m-seg.pt"
    YOLO_IMGSZ = 1280
    YOLO_BATCH = 4
    YOLO_EPOCHS = 70
    YOLO_PATIENCE = 70
    YOLO_MASK_RATIO = 2
    YOLO_CLS = 0.5

    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    PREDICT_CONF_FLOOR = 0.08
    PRIMARY_CONF = 0.26
    CONF_SWEEP = (0.20, 0.24, 0.26, 0.28, 0.32)
    PREDICT_IOU = 0.55
    REFINER_THRESHOLD = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    SMALL_GT_MAX_AREA = 1024.0
    SPARSE_R = 0.77
    BOOTSTRAP_ITERS = 2000

for p in (CFG.LOCAL_ROOT, CFG.PREPROC_DIR, CFG.EXP_DIR, CFG.YOLO_DS_ROOT):
    p.mkdir(parents=True, exist_ok=True)

random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
assert torch.cuda.is_available(), "GPU ランタイムを選択してください"
RUNTIME_INFO = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0),
    "split_mode": CFG.SPLIT_MODE,
    "sib_neg_weight": CFG.SIB_NEG_WEIGHT,
    "python": sys.version,
    "torch": torch.__version__,
    "ultralytics": version("ultralytics"),
}
print(json.dumps(RUNTIME_INFO, indent=2, ensure_ascii=False))


In [ ]:
# [2/7] MediSegEdgeAttUNet + Sibling Negative Shielding YOLO トレーナー層
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)

"""Cross-annotator unlabeled-sibling negative shielding for YOLO11m-seg (Idea 7, v131).

Mathematical motivation:
In MAGFiLO, 296/707 images have R in {2, 3} annotators, and 57.9% of physical filaments
are singletons (marked by k=1 of R annotators). Under per-record replication (prod), a
k-of-R filament receives k positive updates with soft TAL target t_+ (~0.50) and (R - k)
hard background negative updates (t = 0, weight = 1.0), driving its stationary confidence
to p* = (k * t_+) / (k + (R - k) * w_sib):
  - With w_sib = 1.0 (prod): R=2, k=1 -> p* ~ 0.25 < 0.26 (dropped right at the 0.26 cliff,
    even though RESEARCH_LOG §2.66 proved R=2, k=1 predictions have strongly positive
    marginal PQ = 0.65 >> 0.445!).
  - With w_sib = 0.20 (80% sibling negative shielding):
      * R=1, k=1 -> p* ~ 0.50
      * R=2, k=1 -> p* ~ 0.42 >= 0.26 (recovered!)
      * R=3, k=2 -> p* ~ 0.45 >= 0.26
      * R=3, k=1 -> p* ~ 0.36 (strictly preserving consensus order 3/3 > 2/3 > 1/2 > 1/3)
      * True non-filament solar disk background (k=0) keeps w_bce = 1.0 across all R copies!

Unlike v87 (union_hard / union_agree), positive polygons remain 100% crisp single-annotator
masks (no union bloat), and pseudo-class cls=1 polygons pass through Ultralytics's exact
Mosaic + RandomPerspective + Flip pipeline before SplitSiblingPolygons strips them prior to
Format so the model stays strictly nc=1.
"""
from typing import Any

import cv2
import numpy as np
import torch


def _format_poly_line(cls_id: int, poly_xy: np.ndarray, img_h: int, img_w: int) -> str | None:
    pts = np.asarray(poly_xy, dtype=np.float64).reshape(-1, 2)
    if len(pts) < 2:
        return None
    if len(pts) == 2:
        mid = 0.5 * (pts[0:1] + pts[1:2])
        pts = np.vstack([pts[0:1], mid, pts[1:2]])
    xs = np.clip(pts[:, 0] / float(img_w), 0.0, 1.0)
    ys = np.clip(pts[:, 1] / float(img_h), 0.0, 1.0)
    interleaved = np.empty(2 * len(pts), dtype=np.float64)
    interleaved[0::2] = xs
    interleaved[1::2] = ys
    coords = " ".join(f"{v:.6f}" for v in interleaved)
    return f"{cls_id} {coords}"


def emit_sibling_shield_label_txt(
    pos_polys_xy: list[np.ndarray],
    sibling_polys_xy: list[np.ndarray],
    img_h: int,
    img_w: int,
) -> str:
    """Emit Ultralytics seg .txt with cls=0 (annotator GT) and cls=1 (sibling ignore)."""
    lines: list[str] = []
    for poly in pos_polys_xy:
        ln = _format_poly_line(0, poly, img_h, img_w)
        if ln is not None:
            lines.append(ln)
    for sib in sibling_polys_xy:
        ln = _format_poly_line(1, sib, img_h, img_w)
        if ln is not None:
            lines.append(ln)
    return "\n".join(lines) + ("\n" if lines else "")


class SplitSiblingPolygons:
    """Transform inserted right before Format to strip cls=1 sibling-ignore polygons."""

    def __init__(self, sibling_cls: int = 1):
        self.sibling_cls = int(sibling_cls)

    def __call__(self, labels: dict[str, Any]) -> dict[str, Any]:
        cls_arr = labels.get("cls")
        instances = labels.get("instances")
        if cls_arr is None or instances is None or len(cls_arr) == 0:
            labels["sibling_polys_px"] = []
            return labels

        flat_cls = np.asarray(cls_arr).reshape(-1)
        is_sib = flat_cls == self.sibling_cls
        if not np.any(is_sib):
            labels["sibling_polys_px"] = []
            return labels

        h, w = labels["img"].shape[:2]
        if getattr(instances, "normalized", False):
            instances.denormalize(w, h)
        segs = instances.segments
        sib_polys = [np.asarray(segs[i], dtype=np.float32).copy() for i in np.flatnonzero(is_sib)]

        keep = ~is_sib
        labels["instances"] = instances[keep]
        kept_cls = flat_cls[keep].astype(np.float32)
        labels["cls"] = kept_cls.reshape(-1, 1) if cls_arr.ndim == 2 else kept_cls
        labels["sibling_polys_px"] = sib_polys
        return labels


def rasterize_sibling_grid(
    sib_polys_px: list[np.ndarray],
    grid_h: int,
    grid_w: int,
    img_h: int,
    img_w: int,
    *,
    dilate_cells: int = 1,
) -> np.ndarray:
    """Rasterize pixel-space sibling polygons onto a ``(grid_h, grid_w)`` bool mask."""
    canvas = np.zeros((grid_h, grid_w), dtype=np.uint8)
    if not sib_polys_px:
        return canvas.astype(bool)
    sx = float(grid_w) / max(float(img_w), 1.0)
    sy = float(grid_h) / max(float(img_h), 1.0)
    valid_ipts = []
    for poly in sib_polys_px:
        pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
        if len(pts) < 3:
            continue
        mapped = np.empty_like(pts)
        mapped[:, 0] = pts[:, 0] * sx
        mapped[:, 1] = pts[:, 1] * sy
        valid_ipts.append(np.round(mapped).astype(np.int32).reshape(-1, 1, 2))
    if valid_ipts:
        cv2.fillPoly(canvas, valid_ipts, color=1)
        if dilate_cells > 0:
            k = 2 * int(dilate_cells) + 1
            canvas = cv2.dilate(canvas, np.ones((k, k), dtype=np.uint8), iterations=1)
    return canvas > 0


def compute_sibling_bce_weights(
    sibling_batch: list[list[np.ndarray]] | tuple[list[np.ndarray], ...],
    anchor_points_px: torch.Tensor,
    fg_mask: torch.Tensor,
    imgsz_hw: tuple[int, int],
    *,
    grid_hw: tuple[int, int] | None = None,
    sib_neg_weight: float = 0.20,
    dilate_cells: int = 1,
) -> torch.Tensor:
    """Compute ``(B, N_anchors)`` BCE weights shielding ``~fg_mask`` anchors inside sibling masks.

    - Positive anchors (``fg_mask == True``) ALWAYS receive weight ``1.0``.
    - True background anchors outside ``sibling_batch`` ALWAYS receive weight ``1.0``.
    - ``~fg_mask`` anchors whose center falls inside a sibling annotator's polygon receive
      ``sib_neg_weight`` (e.g. ``0.20``).
    - When ``sib_neg_weight == 1.0`` or ``sibling_batch`` has no polygons, returns all ones.
    """
    bsz, n_anc = fg_mask.shape
    weights = torch.ones((bsz, n_anc), dtype=torch.float32, device=fg_mask.device)
    if abs(float(sib_neg_weight) - 1.0) < 1e-7 or not sibling_batch:
        return weights
    if not any(len(p) > 0 for p in sibling_batch):
        return weights

    img_h, img_w = int(imgsz_hw[0]), int(imgsz_hw[1])
    if grid_hw is None:
        gh, gw = max(1, img_h // 8), max(1, img_w // 8)
    else:
        gh, gw = int(grid_hw[0]), int(grid_hw[1])

    grids = np.stack(
        [
            rasterize_sibling_grid(
                sibling_batch[b] if b < len(sibling_batch) else [],
                gh,
                gw,
                img_h,
                img_w,
                dilate_cells=dilate_cells,
            )
            for b in range(bsz)
        ],
        axis=0,
    )
    grid_t = torch.from_numpy(grids).to(device=fg_mask.device, dtype=torch.bool)

    # anchor_points_px: (N_anchors, 2) in (x_px, y_px)
    sx = float(gw) / max(float(img_w), 1.0)
    sy = float(gh) / max(float(img_h), 1.0)
    cx = (anchor_points_px[:, 0] * sx).long().clamp(0, gw - 1)
    cy = (anchor_points_px[:, 1] * sy).long().clamp(0, gh - 1)

    in_sib = grid_t[:, cy, cx]  # (B, N_anchors)
    shield_mask = (~fg_mask) & in_sib
    weights[shield_mask] = float(sib_neg_weight)
    return weights


def build_sibling_shield_trainer_classes(
    *,
    sib_neg_weight: float = 0.20,
    dilate_cells: int = 1,
):
    """Build Ultralytics Dataset, Loss, and SegmentationTrainer with sibling negative shielding."""
    from torch import nn
    from ultralytics.data.augment import Format
    from ultralytics.data.dataset import YOLODataset
    from ultralytics.models.yolo.segment import SegmentationTrainer
    from ultralytics.utils.loss import v8SegmentationLoss
    from ultralytics.utils.tal import make_anchors
    from ultralytics.utils.torch_utils import unwrap_model

    class SiblingShieldYOLODataset(YOLODataset):
        def build_transforms(self, hyp=None):
            transforms = super().build_transforms(hyp=hyp)
            inserted = False
            for idx, t in enumerate(transforms.transforms):
                if isinstance(t, Format):
                    transforms.transforms.insert(idx, SplitSiblingPolygons(sibling_cls=1))
                    inserted = True
                    break
            if not inserted:
                transforms.transforms.append(SplitSiblingPolygons(sibling_cls=1))
            return transforms

        @staticmethod
        def collate_fn(batch: list[dict]) -> dict:
            sib_list = [sample.pop("sibling_polys_px", []) for sample in batch]
            collated = YOLODataset.collate_fn(batch)
            collated["sibling_polys_px"] = sib_list
            return collated

    class SiblingShieldSegmentationLoss(v8SegmentationLoss):
        def __init__(self, model: nn.Module, tal_topk: int = 10):
            super().__init__(model, tal_topk=tal_topk)
            self.sib_neg_weight = float(sib_neg_weight)
            self.dilate_cells = int(dilate_cells)

        def get_assigned_targets_and_loss(self, preds: dict, batch: dict) -> tuple:
            loss = torch.zeros(3, device=self.device)  # box, cls, dfl
            pred_distri, pred_scores = (
                preds["boxes"].permute(0, 2, 1).contiguous(),
                preds["scores"].permute(0, 2, 1).contiguous(),
            )
            anchor_points, stride_tensor = make_anchors(preds["feats"], self.stride, 0.5)

            dtype = pred_scores.dtype
            batch_size = pred_scores.shape[0]
            imgsz = (
                torch.tensor(preds["feats"][0].shape[2:], device=self.device, dtype=dtype)
                * self.stride[0]
            )

            targets = torch.cat(
                (batch["batch_idx"].view(-1, 1), batch["cls"].view(-1, 1), batch["bboxes"]), 1
            )
            targets = self.preprocess(
                targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]]
            )
            gt_labels, gt_bboxes = targets.split((1, 4), 2)
            mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0.0)

            pred_bboxes = self.bbox_decode(anchor_points, pred_distri)
            anchor_points_px = anchor_points * stride_tensor

            _, target_bboxes, target_scores, fg_mask, target_gt_idx = self.assigner(
                pred_scores.detach().sigmoid(),
                (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
                anchor_points_px,
                gt_labels,
                gt_bboxes,
                mask_gt,
            )

            p3_h, p3_w = int(preds["feats"][0].shape[2]), int(preds["feats"][0].shape[3])
            img_h, img_w = round(float(imgsz[0].item())), round(float(imgsz[1].item()))
            w_bce = compute_sibling_bce_weights(
                batch.get("sibling_polys_px", ()),
                anchor_points_px.detach(),
                fg_mask,
                (img_h, img_w),
                grid_hw=(p3_h, p3_w),
                sib_neg_weight=self.sib_neg_weight,
                dilate_cells=self.dilate_cells,
            )

            target_scores_sum = max(target_scores.sum(), 1)
            bce_loss = self.bce(pred_scores, target_scores.to(dtype)) * w_bce.to(dtype).unsqueeze(
                -1
            )
            if self.class_weights is not None:
                bce_loss = bce_loss * self.class_weights
            loss[1] = bce_loss.sum() / target_scores_sum

            if fg_mask.sum():
                loss[0], loss[2] = self.bbox_loss(
                    pred_distri,
                    pred_bboxes,
                    anchor_points,
                    target_bboxes / stride_tensor,
                    target_scores,
                    target_scores_sum,
                    fg_mask,
                    imgsz,
                    stride_tensor,
                )

            loss[0] *= self.hyp.box
            loss[1] *= self.hyp.cls
            loss[2] *= self.hyp.dfl
            return (
                (fg_mask, target_gt_idx, target_bboxes, anchor_points, stride_tensor),
                loss,
                dict(zip(self.loss_names, loss.detach())),
            )

    class SiblingShieldSegmentationTrainer(SegmentationTrainer):
        def get_model(self, cfg=None, weights=None, verbose: bool = True):
            self.data["nc"] = 1
            self.data["names"] = {0: "filament"}
            model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)
            return model

        def set_model_attributes(self):
            super().set_model_attributes()
            umodel = unwrap_model(self.model)
            umodel.nc = 1
            umodel.names = {0: "filament"}
            umodel.criterion = SiblingShieldSegmentationLoss(umodel)

        def build_dataset(self, img_path: str, mode: str = "train", batch: int | None = None):
            gs = max(int(unwrap_model(self.model).stride.max() if self.model else 0), 32)
            data_for_loader = dict(self.data)
            data_for_loader["nc"] = 2
            data_for_loader["names"] = {0: "filament", 1: "sibling_ignore"}
            return SiblingShieldYOLODataset(
                img_path=img_path,
                imgsz=self.args.imgsz,
                batch_size=batch,
                augment=mode == "train",
                hyp=self.args,
                rect=self.args.rect or mode == "val",
                cache=self.args.cache or None,
                single_cls=False,
                stride=int(gs),
                pad=0.0 if mode == "train" else 0.5,
                prefix=f"{mode}: ",
                task=self.args.task,
                classes=self.args.classes,
                data=data_for_loader,
                fraction=self.args.fraction if mode == "train" else 1.0,
            )

    return (
        SiblingShieldYOLODataset,
        SiblingShieldSegmentationLoss,
        SiblingShieldSegmentationTrainer,
    )


In [ ]:
# [3/7] COCO 読込・台帳 SHA256 検証・分割固定・アノテータ間合意クラスタ計算
PINNED_EXCLUSIONS_CSV = """file_name,time_block_role,fold,exclusion_reason
20161229133734Ch.jpeg,development,0,within_5d_of_locked_eval
20161231133734Ch.jpeg,development,1,within_5d_of_locked_eval
20170102024454Uh.jpeg,locked_evaluation,1,within_5d_of_development
20170103030554Uh.jpeg,locked_evaluation,1,within_5d_of_development
"""

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

TRAIN_JSON = next(Path(CFG.DATA_CACHE).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
coco = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
images = {item["id"]: item for item in coco["images"]}
annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

folds_path = CFG.ARTIFACT_ROOT / "grouped_folds.csv"
folds_sha = file_sha256(folds_path)
if folds_sha != CFG.GROUPED_FOLDS_SHA256:
    raise ValueError(f"grouped_folds.csv SHA256 mismatch: {folds_sha}")
fold_df = pd.read_csv(folds_path)

tb_path = CFG.ARTIFACT_ROOT / "time_block_assignment.csv"
if tb_path.exists():
    tb_sha = file_sha256(tb_path)
    if tb_sha != CFG.TIME_BLOCK_SHA256:
        raise ValueError(f"time_block_assignment.csv SHA256 mismatch: {tb_sha}")
    tb_df = pd.read_csv(tb_path)
else:
    tb_df = fold_df.copy()
    tb_df["year"] = tb_df["file_name"].str.slice(0, 4).astype(int)
    tb_df["time_block_role"] = np.where(tb_df["year"] <= 2016, "development", "locked_evaluation")
    tb_sha = "derived_year_le_2016"

excl_path = CFG.ARTIFACT_ROOT / "spine_aux_prereg_exclusions_2026-11-13.csv"
if excl_path.exists():
    excl_sha = file_sha256(excl_path)
    if excl_sha != CFG.EXCLUSIONS_SHA256:
        raise ValueError(f"exclusions CSV SHA256 mismatch: {excl_sha}")
    excl_df = pd.read_csv(excl_path)
else:
    raw_bytes = PINNED_EXCLUSIONS_CSV.encode("utf-8")
    excl_df = pd.read_csv(io.StringIO(PINNED_EXCLUSIONS_CSV))
    excl_sha = hashlib.sha256(raw_bytes).hexdigest()

excluded_set = set(excl_df["file_name"].tolist())
role_of_file = dict(zip(tb_df["file_name"], tb_df["time_block_role"]))
fold_of_file = dict(zip(fold_df["file_name"], fold_df["fold"].astype(int)))

if CFG.SPLIT_MODE == "pilot":
    fit_files = [f for f in train_files if role_of_file[f] == "development" and fold_of_file[f] in (0, 1) and f not in excluded_set]
    valid_files = [f for f in train_files if role_of_file[f] == "development" and fold_of_file[f] == 2 and f not in excluded_set]
    assert len(fit_files) == 365 and len(valid_files) == 180, (len(fit_files), len(valid_files))
    assert not any(role_of_file[f] == "locked_evaluation" for f in fit_files + valid_files)
elif CFG.SPLIT_MODE == "temporal_confirm":
    fit_files = [f for f in train_files if role_of_file[f] == "development" and f not in excluded_set]
    valid_files = [f for f in train_files if role_of_file[f] == "locked_evaluation"]
    assert len(fit_files) == 545 and len(valid_files) == 160, (len(fit_files), len(valid_files))
else:
    raise ValueError(f"Unknown SPLIT_MODE: {CFG.SPLIT_MODE}")
assert set(fit_files).isdisjoint(valid_files), "fit/eval overlap detected!"

for name in tqdm(sorted(set(fit_files) | set(valid_files)), desc="前処理画像をローカル SSD へコピー"):
    dst = CFG.PREPROC_DIR / name
    if not dst.exists():
        shutil.copy2(CFG.PREPROC_CACHE / name, dst)

def to_rle(annotation, height, width):
    seg = annotation["segmentation"]
    if isinstance(seg, list):
        return mask_utils.merge(mask_utils.frPyObjects(seg, height, width))
    if isinstance(seg.get("counts"), list):
        return mask_utils.frPyObjects(seg, height, width)
    rle = dict(seg)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

gt_rles, gt_areas, gt_polys = {}, {}, {}
for image_id, info in images.items():
    h, w = int(info["height"]), int(info["width"])
    r_list, a_list, p_list = [], [], []
    for a in annotations[image_id]:
        if a.get("iscrowd", 0):
            continue
        segs = [p for p in a.get("segmentation", []) if len(p) >= 6]
        if not segs:
            continue
        r = to_rle(a, h, w)
        r_list.append(r)
        a_list.append(float(a.get("area", mask_utils.area(r))))
        p_list.append(np.asarray(max(segs, key=len), dtype=np.float64).reshape(-1, 2))
    gt_rles[image_id] = r_list
    gt_areas[image_id] = np.asarray(a_list, dtype=np.float64)
    gt_polys[image_id] = p_list

# 各 GT が multi-annotator 画像上で consensus (他 record と IoU>0.5 一致あり) か singleton かを事前ラベル付け (診断用)
gt_is_consensus = {}
for fname in valid_files:
    iids = file_to_ids[fname]
    if len(iids) == 1:
        gt_is_consensus[iids[0]] = np.zeros(len(gt_rles[iids[0]]), dtype=bool)
        continue
    for iid in iids:
        my_rles = gt_rles[iid]
        other_rles = [r for oid in iids if oid != iid for r in gt_rles[oid]]
        if not my_rles or not other_rles:
            gt_is_consensus[iid] = np.zeros(len(my_rles), dtype=bool)
        else:
            mat = np.asarray(mask_utils.iou(other_rles, my_rles, [0] * len(my_rles)), dtype=np.float64).T
            gt_is_consensus[iid] = mat.max(axis=1) > 0.5

print({"split_mode": CFG.SPLIT_MODE, "fit_obs": len(fit_files), "eval_obs": len(valid_files),
       "folds_sha": folds_sha[:12], "tb_sha": tb_sha[:12], "excl_sha": excl_sha[:12]})


In [ ]:
# [4/7] Sibling-Shield データセット構築 & 学習前パリティ検査
def build_sibling_dataset():
    ds_root = CFG.YOLO_DS_ROOT / f"{CFG.SPLIT_MODE}_fit{len(fit_files)}_val{len(valid_files)}"
    done_marker = ds_root / "DATASET_READY_V131"
    local_ds = CFG.LOCAL_ROOT / f"yolo_ds_{CFG.SPLIT_MODE}"

    if not done_marker.exists():
        stats = {"train_samples": 0, "pos_instances": 0, "sibling_ignore_instances": 0, "val_samples": 0}
        for fname in tqdm(fit_files, desc="train 画像+sibling_ignoreラベル生成"):
            iids = file_to_ids[fname]
            for iid in iids:
                info = images[iid]
                h, w = int(info["height"]), int(info["width"])
                stem = f"{Path(fname).stem}__labeler_{iid}"
                img_out = ds_root / "images" / "train" / f"{stem}.jpeg"
                lbl_out = ds_root / "labels" / "train" / f"{stem}.txt"
                img_out.parent.mkdir(parents=True, exist_ok=True)
                lbl_out.parent.mkdir(parents=True, exist_ok=True)
                if not img_out.exists():
                    shutil.copy2(CFG.PREPROC_DIR / fname, img_out)

                pos_polys = gt_polys[iid]
                pos_rles = gt_rles[iid]
                sib_polys = []
                # 同一物理画像の他アノテータが描いた個体のうち、自身の GT と IoU <= 0.30 のものを抽出
                for oid in iids:
                    if oid == iid:
                        continue
                    for o_rle, o_poly in zip(gt_rles[oid], gt_polys[oid]):
                        if not pos_rles:
                            sib_polys.append(o_poly)
                        else:
                            ious = np.asarray(mask_utils.iou(pos_rles, [o_rle], [0]), dtype=np.float64).reshape(-1)
                            if float(ious.max()) <= CFG.SIB_OVERLAP_MAX_IOU:
                                sib_polys.append(o_poly)
                lbl_out.write_text(emit_sibling_shield_label_txt(pos_polys, sib_polys, h, w), encoding="utf-8")
                stats["train_samples"] += 1
                stats["pos_instances"] += len(pos_polys)
                stats["sibling_ignore_instances"] += len(sib_polys)

        for fname in tqdm(valid_files, desc="val 画像+ラベル生成"):
            iid = file_to_ids[fname][0]
            info = images[iid]
            h, w = int(info["height"]), int(info["width"])
            stem = Path(fname).stem
            img_out = ds_root / "images" / "val" / f"{stem}.jpeg"
            lbl_out = ds_root / "labels" / "val" / f"{stem}.txt"
            img_out.parent.mkdir(parents=True, exist_ok=True)
            lbl_out.parent.mkdir(parents=True, exist_ok=True)
            if not img_out.exists():
                shutil.copy2(CFG.PREPROC_DIR / fname, img_out)
            lbl_out.write_text(emit_sibling_shield_label_txt(gt_polys[iid], [], h, w), encoding="utf-8")
            stats["val_samples"] += 1
        (ds_root / "dataset_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
        done_marker.write_text("ok", encoding="utf-8")

    if not (local_ds / "DATASET_READY_V131").exists():
        if local_ds.exists():
            shutil.rmtree(local_ds)
        shutil.copytree(ds_root, local_ds)
    yaml_local = local_ds / "data.yaml"
    yaml_local.write_text(
        f"path: {local_ds.as_posix()}\ntrain: images/train\nval: images/val\nnc: 2\n"
        "names:\n  0: filament\n  1: sibling_ignore\n",
        encoding="utf-8",
    )
    stats = json.loads((local_ds / "dataset_stats.json").read_text(encoding="utf-8"))
    return yaml_local, stats

YAML_PATH, DS_STATS = build_sibling_dataset()
print("Dataset stats:", DS_STATS)

def verify_sibling_shield_sanity():
    from ultralytics.cfg import get_cfg
    from ultralytics.nn.tasks import SegmentationModel
    from ultralytics.utils.loss import v8SegmentationLoss
    torch.manual_seed(CFG.SEED)
    m = SegmentationModel("yolo11n-seg.yaml", ch=3, nc=1, verbose=False).cuda()
    m.args = get_cfg(overrides={"task": "segment", "overlap_mask": False})
    _, sib1_cls, _ = build_sibling_shield_trainer_classes(sib_neg_weight=1.0, dilate_cells=CFG.SIB_DILATE_CELLS)
    _, sibw_cls, _ = build_sibling_shield_trainer_classes(sib_neg_weight=CFG.SIB_NEG_WEIGHT, dilate_cells=CFG.SIB_DILATE_CELLS)
    l_orig = v8SegmentationLoss(m)
    l_sib1 = sib1_cls(m)
    l_sibw = sibw_cls(m)
    x = torch.randn(2, 3, 256, 256, device="cuda")
    masks = torch.zeros((2, 64, 64), dtype=torch.float32, device="cuda")
    masks[0, 8:28, 8:28] = 1.0
    masks[1, 8:28, 8:28] = 1.0
    sib_poly = np.array([[120, 120], [220, 120], [220, 220], [120, 220]], dtype=np.float32)
    batch = {
        "img": x,
        "batch_idx": torch.tensor([0.0, 1.0], device="cuda"),
        "cls": torch.tensor([[0.0], [0.0]], device="cuda"),
        "bboxes": torch.tensor([[0.28, 0.28, 0.30, 0.30], [0.28, 0.28, 0.30, 0.30]], device="cuda"),
        "masks": masks,
        "sibling_polys_px": [[sib_poly], []],
    }
    with torch.amp.autocast("cuda"):
        preds = m(x)
        t0, _ = l_orig(preds, batch)
        t1, _ = l_sib1(preds, batch)
        tw, _ = l_sibw(preds, batch)
    diff = float((t0 - t1).abs().max().item())
    assert diff < 1e-5, f"Parity mismatch at sib_neg_weight=1.0: {diff}"
    assert float(tw[2]) < float(t1[2]), "Sibling shielding did not reduce negative cls loss!"
    del m, l_orig, l_sib1, l_sibw, x, masks, preds; torch.cuda.empty_cache()
    print(f"[Pre-train Sibling-Shield Sanity OK] parity_diff={diff:.2e}")
    return diff

PARITY_DIFF = verify_sibling_shield_sanity()


In [ ]:
# [5/7] Sibling-Shield YOLO11m-seg 70ep 学習 (単一 Arm のみ実行)
def train_sibling_shield_arm():
    run_name = f"yolo_{CFG.SPLIT_MODE}_arm_sibling_shield"
    run_dir = CFG.EXP_DIR / run_name
    last_pt = run_dir / "weights" / "last.pt"
    done_flag = run_dir / "TRAINING_COMPLETE"
    if done_flag.exists() and last_pt.exists():
        print(f"再利用: Sibling-Shield 完了済み重み {last_pt}")
        return last_pt, {"reused": True}

    _, _, trainer_cls = build_sibling_shield_trainer_classes(
        sib_neg_weight=CFG.SIB_NEG_WEIGHT, dilate_cells=CFG.SIB_DILATE_CELLS
    )
    t0 = time.time()
    if last_pt.exists() and not done_flag.exists():
        print(f"中断再開: {last_pt} から resume=True")
        trainer = trainer_cls(overrides={"model": str(last_pt), "resume": True})
    else:
        random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
        trainer = trainer_cls(overrides={
            "model": CFG.YOLO_MODEL,
            "data": str(YAML_PATH),
            "epochs": CFG.YOLO_EPOCHS,
            "imgsz": CFG.YOLO_IMGSZ,
            "batch": CFG.YOLO_BATCH,
            "device": 0,
            "workers": 4,
            "patience": CFG.YOLO_PATIENCE,
            "seed": CFG.SEED,
            "deterministic": True,
            "single_cls": False,
            "mask_ratio": CFG.YOLO_MASK_RATIO,
            "overlap_mask": False,
            "amp": True,
            "cache": "disk",
            "cls": CFG.YOLO_CLS,
            "optimizer": "AdamW",
            "lr0": 7e-4,
            "lrf": 0.05,
            "weight_decay": 5e-4,
            "warmup_epochs": 3,
            "degrees": 180,
            "translate": 0.04,
            "scale": 0.25,
            "fliplr": 0.5,
            "flipud": 0.5,
            "hsv_h": 0.0,
            "hsv_s": 0.0,
            "hsv_v": 0.18,
            "mosaic": 0.20,
            "close_mosaic": 10,
            "project": str(CFG.EXP_DIR),
            "name": run_name,
            "exist_ok": True,
        })
    trainer.train()
    assert last_pt.exists(), f"last.pt missing after training: {last_pt}"
    mins = round((time.time() - t0) / 60.0, 2)
    done_flag.write_text(
        json.dumps({"finished_utc": datetime.now(timezone.utc).isoformat(), "minutes": mins, "sib_neg_weight": CFG.SIB_NEG_WEIGHT}),
        encoding="utf-8",
    )
    torch.cuda.empty_cache()
    return last_pt, {"reused": False, "train_minutes": mins, "sib_neg_weight": CFG.SIB_NEG_WEIGHT}

SIB_SHIELD_PT, SIB_TRAIN_LOG = train_sibling_shield_arm()
print("Sibling-Shield training log:", SIB_TRAIN_LOG)


In [ ]:
# [6/7] 共通 in-split Refiner & 対照 Arm A (v127) 解決
ref_name = f"refiner_edgeatt_{CFG.SPLIT_MODE}_fit{len(fit_files)}"
REFINER_CKPT_PATH = CFG.V127_EXP_DIR / f"{ref_name}.pt"
assert REFINER_CKPT_PATH.exists(), f"Missing fit-only refiner: {REFINER_CKPT_PATH}"

CONTROL_ARM_A_PT = CFG.V127_EXP_DIR / f"yolo_{CFG.SPLIT_MODE}_arm_a" / "weights" / "last.pt"
_arm_a_done = CFG.V127_EXP_DIR / f"yolo_{CFG.SPLIT_MODE}_arm_a" / "TRAINING_COMPLETE"
if not _arm_a_done.exists():
    deadline = time.time() + CFG.PEER_WAIT_TIMEOUT_MIN * 60
    while time.time() < deadline and not _arm_a_done.exists():
        print("Waiting for v127 Arm A TRAINING_COMPLETE...")
        time.sleep(30)
assert CONTROL_ARM_A_PT.exists(), f"Missing Arm A control weights: {CONTROL_ARM_A_PT}"
print("Resolved Refiner & Arm A control:", REFINER_CKPT_PATH.name, CONTROL_ARM_A_PT)


In [ ]:
# [7/7] Sibling-Shield vs Arm A 対応評価・Singleton/Consensus 別 Recall 分解・Bootstrap 保存
def sparse_label_pq(sum_iou, tp, fp, fn, r=0.77):
    den = r * tp + 0.5 * fp + 0.5 * (1 - r) * tp + 0.5 * r * fn
    return float(r * sum_iou / den) if den > 0 else 0.0

def mask_square_box(mask, context=CFG.CROP_CONTEXT, minimum=96):
    h, w = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, w, h
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(h, w))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > w: x0 -= x1 - w; x1 = w
    if y1 > h: y0 -= y1 - h; y1 = h
    return max(0, x0), max(0, y0), min(w, x1), min(h, y1)

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped or self_iou[i, j] <= 0:
                continue
            inter = self_iou[i, j] * (areas[i] + areas[j]) / (1.0 + self_iou[i, j])
            smaller = i if areas[i] <= areas[j] else j
            if inter / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership_prod(masks):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    occupied = np.zeros(masks[0].shape, dtype=bool)
    kept = []
    for idx in sorted(drop_contained_indices(masks), key=lambda i: areas[i]):
        excl = masks[idx] & ~occupied
        if excl.sum() < CFG.MIN_AREA:
            continue
        kept.append(excl); occupied |= excl
    return kept

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

def build_five_day_clusters(obs_names):
    parsed = sorted((datetime.strptime(n[:14], "%Y%m%d%H%M%S"), n) for n in obs_names)
    clusters, cur, prev_dt = [], [parsed[0][1]], parsed[0][0]
    for dt, name in parsed[1:]:
        if (dt - prev_dt).total_seconds() > 5 * 86400:
            clusters.append(cur); cur = [name]
        else:
            cur.append(name)
        prev_dt = dt
    clusters.append(cur)
    return clusters

def paired_bootstrap_pq(per_obs_a, per_obs_b, obs_names, block_mode="obs", n_boot=2000, seed=3407):
    rng = np.random.default_rng(seed)
    units = [[n] for n in obs_names] if block_mode == "obs" else build_five_day_clusters(obs_names)
    n_u = len(units)
    deltas = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        pick = rng.integers(0, n_u, size=n_u)
        sa, sb = [0.0, 0, 0, 0], [0.0, 0, 0, 0]
        for ui in pick:
            for name in units[ui]:
                for k in range(4):
                    sa[k] += per_obs_a[name][k]
                    sb[k] += per_obs_b[name][k]
        den_a = sa[1] + 0.5 * sa[2] + 0.5 * sa[3]
        den_b = sb[1] + 0.5 * sb[2] + 0.5 * sb[3]
        deltas[b] = (sb[0] / den_b if den_b > 0 else 0.0) - (sa[0] / den_a if den_a > 0 else 0.0)
    return {
        "n_units": n_u,
        "block_mode": block_mode,
        "mean_delta": float(deltas.mean()),
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "prob_positive": float((deltas > 0).mean()),
    }

def infer_and_cache_batched(weights_path: Path, refiner_model, label: str):
    yolo_net = YOLO(str(weights_path))
    cache = []
    t0 = time.time()
    for name in tqdm(valid_files, desc=f"{label} 推論+バッチrefine"):
        img_path = CFG.PREPROC_DIR / name
        res = yolo_net.predict(
            str(img_path), imgsz=CFG.YOLO_IMGSZ, conf=CFG.PREDICT_CONF_FLOOR,
            iou=CFG.PREDICT_IOU, agnostic_nms=True, retina_masks=True, verbose=False,
        )[0]
        h, w = res.orig_shape
        masks = []
        if res.masks is not None:
            for m in res.masks.data.detach().cpu().numpy():
                if m.shape != (h, w):
                    m = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR)
                masks.append(m >= 0.5)
        conf = res.boxes.conf.detach().cpu().numpy().astype(np.float32) if res.boxes is not None else np.zeros(0, np.float32)
        image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        raw_rles, ref_rles, ref_areas = [], [], []
        if masks:
            boxes = [mask_square_box(pm) for pm in masks]
            crops = np.stack([
                cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
                for (x0, y0, x1, y1) in boxes
            ], axis=0)
            probs = []
            with torch.no_grad():
                for s in range(0, len(crops), 32):
                    ten = torch.from_numpy(crops[s:s + 32]).float().cuda() / 255.0
                    probs.append(refiner_model(ten).sigmoid()[:, 0].cpu().numpy())
            probs_320 = np.concatenate(probs, axis=0)
            for pm, (x0, y0, x1, y1), p320 in zip(masks, boxes, probs_320):
                raw_rles.append(mask_utils.encode(np.asfortranarray(pm.astype(np.uint8))))
                p_box = cv2.resize(p320, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
                rm = np.zeros_like(pm, dtype=bool)
                rm[y0:y1, x0:x1] = p_box >= CFG.REFINER_THRESHOLD
                rm &= pm
                ref_rles.append(mask_utils.encode(np.asfortranarray(rm.astype(np.uint8))))
                ref_areas.append(float(rm.sum()))
        cache.append({"name": name, "conf": conf, "raw_rles": raw_rles, "ref_rles": ref_rles, "ref_areas": np.asarray(ref_areas, dtype=np.float64)})
    sec_per_img = (time.time() - t0) / max(len(valid_files), 1)
    del yolo_net; torch.cuda.empty_cache()
    return cache, sec_per_img

def score_cache_at_conf(cache, conf_thr: float, sec_per_img: float = 0.0):
    sum_iou, tp, fp, fn = 0.0, 0, 0, 0
    multi_singleton_tot, multi_singleton_hit = 0, 0
    multi_consensus_tot, multi_consensus_hit = 0, 0
    small_tot, small_hit = 0, 0
    per_obs = {}
    for row in cache:
        name = row["name"]
        keep_idx = [i for i in range(len(row["conf"])) if row["conf"][i] >= conf_thr and row["ref_areas"][i] >= CFG.MIN_AREA]
        kept_masks = [mask_utils.decode(row["ref_rles"][i]) != 0 for i in keep_idx]
        final_masks = apply_ownership_prod(kept_masks)
        pred_rles = mask_utils.encode(np.asfortranarray(np.stack(final_masks, -1).astype(np.uint8))) if final_masks else []
        obs_stat = [0.0, 0, 0, 0]
        iids = file_to_ids[name]
        for iid in iids:
            truth = gt_rles[iid]
            areas = gt_areas[iid]
            is_cons = gt_is_consensus[iid]
            mat = iou_matrix(truth, pred_rles)
            if len(truth) == 0:
                fp += len(pred_rles); obs_stat[2] += len(pred_rles)
            elif len(pred_rles) == 0:
                fn += len(truth); obs_stat[3] += len(truth)
            else:
                hits = mat > 0.5
                s_i = float(mat[hits].sum())
                t_i = int(hits.sum())
                fp_i = int((hits.sum(axis=0) == 0).sum())
                fn_i = int((hits.sum(axis=1) == 0).sum())
                sum_iou += s_i; tp += t_i; fp += fp_i; fn += fn_i
                obs_stat[0] += s_i; obs_stat[1] += t_i; obs_stat[2] += fp_i; obs_stat[3] += fn_i
            gt_covered = (mat.max(axis=1) > 0.5) if (len(truth) > 0 and len(pred_rles) > 0) else np.zeros(len(truth), dtype=bool)
            if len(iids) >= 2 and len(truth) > 0:
                multi_singleton_tot += int((~is_cons).sum())
                multi_singleton_hit += int((gt_covered & ~is_cons).sum())
                multi_consensus_tot += int(is_cons.sum())
                multi_consensus_hit += int((gt_covered & is_cons).sum())
            if len(truth) > 0:
                sm = areas <= CFG.SMALL_GT_MAX_AREA
                small_tot += int(sm.sum())
                small_hit += int((gt_covered & sm).sum())
        per_obs[name] = tuple(obs_stat)
    den = tp + 0.5 * fp + 0.5 * fn
    pq = sum_iou / den if den > 0 else 0.0
    sq = sum_iou / tp if tp else 0.0
    rq = tp / den if den > 0 else 0.0
    n_records = sum(len(file_to_ids[n]) for n in valid_files)
    return {
        "pq": float(pq),
        "sq": float(sq),
        "rq": float(rq),
        "pq_sparse_r077": sparse_label_pq(sum_iou, tp, fp, fn, CFG.SPARSE_R),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "fp_per_record": fp / max(n_records, 1),
        "multi_singleton_recall": multi_singleton_hit / max(multi_singleton_tot, 1),
        "multi_consensus_recall": multi_consensus_hit / max(multi_consensus_tot, 1),
        "small_recall_final": small_hit / max(small_tot, 1),
        "sec_per_image": round(sec_per_img, 4),
    }, per_obs

refiner = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
_rck = torch.load(REFINER_CKPT_PATH, map_location="cuda", weights_only=False)
refiner.load_state_dict(_rck["best_model"], strict=True)
del _rck

cache_a, sec_a = infer_and_cache_batched(CONTROL_ARM_A_PT, refiner, "arm_a_ctrl_last70")
cache_sib, sec_sib = infer_and_cache_batched(SIB_SHIELD_PT, refiner, "arm_sibling_shield_last70")

res_a, obs_a = score_cache_at_conf(cache_a, CFG.PRIMARY_CONF, sec_a)
res_sib, obs_sib = score_cache_at_conf(cache_sib, CFG.PRIMARY_CONF, sec_sib)

bmode = "obs" if CFG.SPLIT_MODE == "pilot" else "5d_cluster"
boot = paired_bootstrap_pq(obs_a, obs_sib, valid_files, block_mode=bmode, n_boot=CFG.BOOTSTRAP_ITERS, seed=CFG.SEED)
delta_pq = res_sib["pq"] - res_a["pq"]
delta_sparse_pq = res_sib["pq_sparse_r077"] - res_a["pq_sparse_r077"]
fp_rel_change = (res_sib["fp_per_record"] - res_a["fp_per_record"]) / max(res_a["fp_per_record"], 1e-9)

gate_pass = bool((delta_pq >= 0.006) and (boot["ci95_low"] > 0.0)) if CFG.SPLIT_MODE != "pilot" else bool(delta_pq >= 0.008)

sweep_a = {f"conf_{c:.2f}": score_cache_at_conf(cache_a, c, sec_a)[0] for c in CFG.CONF_SWEEP}
sweep_sib = {f"conf_{c:.2f}": score_cache_at_conf(cache_sib, c, sec_sib)[0] for c in CFG.CONF_SWEEP}

manifest_out = {
    "experiment_id": f"v131_sibling_shield_{CFG.SPLIT_MODE}",
    "runtime": RUNTIME_INFO,
    "hashes": {
        "annotations_sha256": file_sha256(TRAIN_JSON),
        "grouped_folds_sha256": folds_sha,
        "time_block_sha256": tb_sha,
        "exclusions_sha256": excl_sha,
        "refiner_sha256": file_sha256(REFINER_CKPT_PATH),
        "arm_a_last_sha256": file_sha256(CONTROL_ARM_A_PT),
        "arm_sibling_shield_last_sha256": file_sha256(SIB_SHIELD_PT),
    },
    "dataset_stats": DS_STATS,
    "arm_a_control": res_a,
    "arm_sibling_shield": res_sib,
    "comparison": {
        "delta_pq_at_026": delta_pq,
        "delta_sparse_pq_r077": delta_sparse_pq,
        "delta_singleton_recall_pt": 100.0 * (res_sib["multi_singleton_recall"] - res_a["multi_singleton_recall"]),
        "delta_consensus_recall_pt": 100.0 * (res_sib["multi_consensus_recall"] - res_a["multi_consensus_recall"]),
        "fp_rel_change": fp_rel_change,
        "bootstrap": boot,
        "gate_pass": gate_pass,
    },
    "conf_sweep_arm_a": sweep_a,
    "conf_sweep_sibling_shield": sweep_sib,
}
manifest_path = CFG.ARTIFACT_ROOT / f"v131_sibling_shield_{CFG.SPLIT_MODE}_manifest.json"
manifest_path.write_text(json.dumps(manifest_out, indent=2, ensure_ascii=False), encoding="utf-8")

print("=== v131 PRIMARY COMPARISON (Sibling-Shield w_sib=0.20 vs Arm A @ conf=0.26, last.pt) ===")
print(f"Arm A (ctrl)     | PQ={res_a['pq']:.5f} sparsePQ={res_a['pq_sparse_r077']:.5f} | TP={res_a['tp']} FP={res_a['fp']} FN={res_a['fn']} | sing_R={res_a['multi_singleton_recall']:.4f} cons_R={res_a['multi_consensus_recall']:.4f}")
print(f"Sibling-Shield   | PQ={res_sib['pq']:.5f} sparsePQ={res_sib['pq_sparse_r077']:.5f} | TP={res_sib['tp']} FP={res_sib['fp']} FN={res_sib['fn']} | sing_R={res_sib['multi_singleton_recall']:.4f} cons_R={res_sib['multi_consensus_recall']:.4f}")
print(json.dumps(manifest_out["comparison"], indent=2, ensure_ascii=False))
print("=== CONFIDENCE SWEEP (PQ / sparsePQ) ===")
for c in CFG.CONF_SWEEP:
    k = f"conf_{c:.2f}"
    print(f"  conf={c:.2f} | Arm A={sweep_a[k]['pq']:.5f} | SibShield={sweep_sib[k]['pq']:.5f} ({sweep_sib[k]['pq'] - sweep_a[k]['pq']:+.5f}) | ΔsparsePQ={sweep_sib[k]['pq_sparse_r077'] - sweep_a[k]['pq_sparse_r077']:+.5f}")
print("Saved manifest:", manifest_path)
